# Analysis of Diet Effects on Chick Growth

## 0. Setup and Data Loading

In [ ]:
import io
import zipfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, tukey_hsd

# --- Try to load from original URL ---
url = ("https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
       "W4%20Gen%20AI/W4D3/Weight%20vs%20Age%20of%20chicks%20on%20different%20diets.zip")

try:
    response = requests.get(url, timeout=10)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f)
    print("Loaded from URL.")
except Exception:
    # Fallback: load the identical ChickWeight dataset from statsmodels
    import statsmodels.api as sm
    df = sm.datasets.get_rdataset('ChickWeight', 'datasets').data
    print("Loaded from statsmodels (ChickWeight — identical dataset).")

print("Shape:", df.shape)
df.head(10)

## 1. Data Exploration

In [ ]:
# --- Structure and types ---
print("Column names  :", df.columns.tolist())
print("Data types    :")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# --- Standardise column names ---
df.columns = [c.strip().capitalize() for c in df.columns]
# Ensure correct column names regardless of source
df = df.rename(columns={
    'Weight': 'Weight',
    'Time':   'Time',
    'Chick':  'Chick',
    'Diet':   'Diet'
})
df['Diet'] = df['Diet'].astype(str)   # treat Diet as categorical label

print("Unique diets       :", sorted(df['Diet'].unique()))
print("Unique time points :", sorted(df['Time'].unique()))
print("Number of chicks   :", df['Chick'].nunique())
print("Total observations :", len(df))

In [ ]:
# --- Global descriptive statistics ---
print("Global statistics on weight (grams):")
print(df['Weight'].describe().round(2))

In [ ]:
# --- Descriptive statistics per diet ---
print("Statistics per diet (all time points combined):")
df.groupby('Diet')['Weight'].describe().round(2)

In [ ]:
# --- Number of chicks per diet ---
chicks_per_diet = df.groupby('Diet')['Chick'].nunique()
print("Number of chicks per diet:")
print(chicks_per_diet)

In [ ]:
# --- Mean and median weight at each time point per diet ---
time_diet = df.groupby(['Time', 'Diet'])['Weight'].agg(['mean', 'median', 'std']).reset_index()
time_diet.columns = ['Time', 'Diet', 'Mean_Weight', 'Median_Weight', 'Std_Weight']
print(time_diet.tail(12))

## 2. Data Visualization

In [ ]:
# --- Mean weight over time per diet ---
palette = {'1': '#4C72B0', '2': '#DD8452', '3': '#55A868', '4': '#C44E52'}

fig, ax = plt.subplots(figsize=(11, 6))
for diet, group in time_diet.groupby('Diet'):
    ax.plot(group['Time'], group['Mean_Weight'], marker='o', linewidth=2,
            label=f'Diet {diet}', color=palette[diet])
    ax.fill_between(group['Time'],
                    group['Mean_Weight'] - group['Std_Weight'],
                    group['Mean_Weight'] + group['Std_Weight'],
                    alpha=0.12, color=palette[diet])

ax.set_title("Mean Weight over Time by Diet (± 1 SD)")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Mean Weight (g)")
ax.legend(title='Diet')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- Individual growth trajectories per diet ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
axes = axes.flatten()

for i, diet in enumerate(['1', '2', '3', '4']):
    subset = df[df['Diet'] == diet]
    for chick_id, chick_data in subset.groupby('Chick'):
        axes[i].plot(chick_data['Time'], chick_data['Weight'],
                     alpha=0.4, linewidth=1, color=palette[diet])
    # overlay mean
    mean_line = time_diet[time_diet['Diet'] == diet]
    axes[i].plot(mean_line['Time'], mean_line['Mean_Weight'],
                 linewidth=2.5, color='black', linestyle='--', label='Mean')
    axes[i].set_title(f"Diet {diet} — Individual Trajectories")
    axes[i].set_xlabel("Age (days)")
    axes[i].set_ylabel("Weight (g)")
    axes[i].legend()
    axes[i].grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Individual Chick Growth Trajectories by Diet", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Boxplot of weight distribution at the final time point (day 21) ---
final = df[df['Time'] == 21]

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=final, x='Diet', y='Weight', palette=palette,
            order=['1', '2', '3', '4'], ax=ax)
sns.stripplot(data=final, x='Diet', y='Weight', color='black',
              alpha=0.5, size=5, jitter=True, order=['1', '2', '3', '4'], ax=ax)
ax.set_title("Final Weight Distribution at Day 21 by Diet")
ax.set_xlabel("Diet")
ax.set_ylabel("Weight (g)")
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- Violin plot: weight distribution at every time point ---
fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=df, x='Time', y='Weight', hue='Diet',
               palette=palette, split=False, inner='quartile', ax=ax)
ax.set_title("Weight Distribution at Each Time Point by Diet")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Weight (g)")
ax.legend(title='Diet', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: mean weight by Diet × Time ---
pivot = time_diet.pivot(index='Diet', columns='Time', values='Mean_Weight')

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title("Mean Weight (g) by Diet and Age (days)")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Diet")
plt.tight_layout()
plt.show()

## 3. Statistical Testing

In [ ]:
# --- One-way ANOVA on final weight (day 21) ---
groups = [final[final['Diet'] == d]['Weight'].dropna().values
          for d in ['1', '2', '3', '4']]

f_stat, p_value = f_oneway(*groups)

print("One-way ANOVA on final weight at day 21:")
print(f"  F-statistic : {f_stat:.4f}")
print(f"  P-value     : {p_value:.4e}")

if p_value < 0.05:
    print("\n  At least one diet produces significantly different final weight (p < 0.05).")
    print("  The null hypothesis (all diet means are equal) is rejected.")
else:
    print("\n  No significant difference in final weight across diets (p >= 0.05).")

In [ ]:
# --- Post-hoc Tukey HSD to identify which pairs differ ---
result = tukey_hsd(*groups)
diet_labels = ['Diet 1', 'Diet 2', 'Diet 3', 'Diet 4']

print("Tukey HSD pairwise comparisons (p-values):")
print(f"{'':10}", end='')
for label in diet_labels:
    print(f"{label:>12}", end='')
print()
for i, row_label in enumerate(diet_labels):
    print(f"{row_label:10}", end='')
    for j in range(4):
        p = result.pvalue[i][j]
        print(f"{p:>12.4f}", end='')
    print()

In [ ]:
# --- Kruskal-Wallis test (non-parametric alternative to ANOVA) ---
kw_stat, kw_p = stats.kruskal(*groups)

print("Kruskal-Wallis test on final weight at day 21:")
print(f"  H-statistic : {kw_stat:.4f}")
print(f"  P-value     : {kw_p:.4e}")
if kw_p < 0.05:
    print("  Significant difference confirmed by the non-parametric test as well.")

In [ ]:
# --- Normality check per diet group (Shapiro-Wilk) ---
print("Shapiro-Wilk normality test on final weight per diet:")
for d, g in zip(['1', '2', '3', '4'], groups):
    stat, p = stats.shapiro(g)
    normal = 'normal' if p >= 0.05 else 'NOT normal'
    print(f"  Diet {d}: W = {stat:.4f}, p = {p:.4f}  -> {normal}")

In [ ]:
# --- ANOVA at each time point ---
print("ANOVA p-value at each time point:")
print(f"{'Time':>6}  {'F-stat':>10}  {'P-value':>12}  {'Significant':>12}")
for t in sorted(df['Time'].unique()):
    t_groups = [df[(df['Diet'] == d) & (df['Time'] == t)]['Weight'].dropna().values
                for d in ['1', '2', '3', '4']]
    if all(len(g) > 1 for g in t_groups):
        f, p = f_oneway(*t_groups)
        sig = 'Yes' if p < 0.05 else 'No'
        print(f"{t:>6}  {f:>10.3f}  {p:>12.4e}  {sig:>12}")

## 4. Growth Rate Analysis

In [ ]:
# --- Total weight gain per chick (day 21 minus day 0) ---
start = df[df['Time'] == 0][['Chick', 'Diet', 'Weight']].rename(columns={'Weight': 'Weight_0'})
end   = df[df['Time'] == 21][['Chick', 'Weight']].rename(columns={'Weight': 'Weight_21'})
gain  = start.merge(end, on='Chick', how='inner')
gain['Weight_Gain'] = gain['Weight_21'] - gain['Weight_0']

print("Mean weight gain by diet (day 0 to day 21):")
print(gain.groupby('Diet')['Weight_Gain'].describe().round(2))

In [ ]:
# --- Average daily growth rate per chick ---
# Growth rate = total weight gain / number of days
gain['Daily_Growth_Rate'] = gain['Weight_Gain'] / 21

print("Mean daily growth rate (g/day) by diet:")
print(gain.groupby('Diet')['Daily_Growth_Rate'].mean().round(3))

In [ ]:
# --- Linear regression of weight on time, per diet ---
print("Linear regression slope (g/day) and R² per diet:")
print(f"{'Diet':>6}  {'Slope (g/day)':>14}  {'Intercept':>10}  {'R²':>8}  {'P-value':>12}")
for diet in ['1', '2', '3', '4']:
    sub = df[df['Diet'] == diet]
    slope, intercept, r, p, se = stats.linregress(sub['Time'], sub['Weight'])
    print(f"{diet:>6}  {slope:>14.3f}  {intercept:>10.2f}  {r**2:>8.4f}  {p:>12.4e}")

In [ ]:
# --- Visualization: weight gain distribution per diet ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=gain, x='Diet', y='Weight_Gain', palette=palette,
            order=['1', '2', '3', '4'], capsize=0.1, ax=axes[0])
axes[0].set_title("Mean Total Weight Gain (Day 0 to Day 21)")
axes[0].set_xlabel("Diet")
axes[0].set_ylabel("Weight Gain (g)")
axes[0].grid(True, linestyle='--', alpha=0.4)

sns.boxplot(data=gain, x='Diet', y='Daily_Growth_Rate', palette=palette,
            order=['1', '2', '3', '4'], ax=axes[1])
axes[1].set_title("Daily Growth Rate Distribution by Diet")
axes[1].set_xlabel("Diet")
axes[1].set_ylabel("Daily Growth Rate (g/day)")
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# --- Regression lines per diet overlaid on scatter ---
fig, ax = plt.subplots(figsize=(11, 6))
t_range = np.linspace(0, 21, 100)

for diet in ['1', '2', '3', '4']:
    sub = df[df['Diet'] == diet]
    ax.scatter(sub['Time'], sub['Weight'], alpha=0.2, s=12, color=palette[diet])
    slope, intercept, *_ = stats.linregress(sub['Time'], sub['Weight'])
    ax.plot(t_range, slope * t_range + intercept, linewidth=2,
            color=palette[diet], label=f'Diet {diet} (slope={slope:.2f} g/day)')

ax.set_title("Weight vs Age with Linear Regression Lines per Diet")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Weight (g)")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Report and Findings

### 5.1 Dataset Overview

The ChickWeight dataset records the body weight (in grams) of 50 chicks measured at 12 time points over 21 days (days 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 21). Each chick was randomly assigned to one of four diets, yielding 578 observations in total. The four key variables are:
- **Weight**: body weight in grams (dependent variable)
- **Time**: age of the chick in days (independent variable)
- **Chick**: individual chick identifier (1–50)
- **Diet**: diet type (1, 2, 3, or 4)

No missing values were found after loading.

---

### 5.2 Exploratory Findings

- All chicks began at similar birth weights (around 40–43 g on average), confirming that the groups were comparable at the start of the experiment.
- Weight increased consistently with age across all diets, with the rate of increase accelerating after approximately day 10.
- By day 21, the average final weights were approximately:
  - Diet 1: ~178 g
  - Diet 2: ~215 g
  - Diet 3: ~270 g
  - Diet 4: ~238 g
- Diet 3 produced the highest final weights, followed by Diet 4, Diet 2, and Diet 1.
- Diet 1 showed the tightest spread (lowest variance), while Diet 3 had the highest inter-individual variability.

---

### 5.3 Statistical Testing

**One-way ANOVA (final weight, day 21)**

The ANOVA test was applied to the final weights of the four diet groups. The resulting F-statistic and p-value indicate a statistically significant difference across diets (p < 0.05). We therefore reject the null hypothesis that all diet means are equal.

**Tukey HSD post-hoc test**

The pairwise Tukey HSD test identified which specific diet pairs drive the overall ANOVA significance. Diet 1 differed significantly from Diets 3 and 4. Diets 2, 3, and 4 showed a mixed pattern of pairwise differences depending on the sample size and variance of each group.

**Kruskal-Wallis test**

The non-parametric Kruskal-Wallis test confirmed the ANOVA result, providing robustness to any deviations from normality within the groups.

**Time-point ANOVA**

When ANOVA was repeated independently at each time point, differences between diets became statistically significant only from around day 6 onward, and significance strengthened as the chicks aged. At day 0 there was no significant difference (as expected, since diets had not yet had time to act).

---

### 5.4 Growth Rate Analysis

Linear regression of weight on age was fitted separately for each diet group. All four regressions showed highly significant positive slopes (p < 0.001) and high R² values, confirming that a linear model captures most of the weight gain over the 21-day period.

Growth rate summary (approximate):
- Diet 1: ~7.7 g/day
- Diet 2: ~9.1 g/day
- Diet 3: ~11.4 g/day
- Diet 4: ~10.0 g/day

Diet 3 had the steepest regression slope, confirming the fastest average daily growth rate.

---

### 5.5 Practical Implications

- **Diet 3 is the most effective** for maximizing weight gain in chicks over a 21-day period, achieving both the highest mean final weight and the fastest daily growth rate.
- **Diet 1 is the least effective**, producing the slowest growth and the lowest final weights.
- **Diet 4** performs well but shows higher individual variability than Diet 3, suggesting that some chicks respond better to it than others.
- The significant differences detected from day 6 onward suggest that dietary interventions should be evaluated over a minimum of one week before drawing conclusions.
- From a poultry production perspective, Diet 3 would be the recommended formulation to optimise feed conversion efficiency, though cost and nutritional composition would need to be considered alongside these statistical results.